# Load essentials

In [2]:
from glob import glob
from tqdm import tqdm
import json
import pandas as pd
import numpy as np

img_codes = [path.split('\\')[-1].split('.')[0] for path in glob(r'..\additional_data\transcript_img\*.txt')]

# Load paths of images & vids

In [3]:
vid_paths = glob(r'D:\AIC_LITE\vids\*\*')
img_paths = glob(r'D:\AIC_LITE\kfs\*\*\*\*')
img_path_maps = {code: path for code, path in zip(img_codes, img_paths)}
vid_path_maps = {}
for code in img_codes:
    for vid in vid_paths:
        if code[-4] not in vid: continue
        vid_path_maps[code] = vid
        break

# Load metadata

In [4]:
met_paths = glob(r'D:\AIC_LITE\metadata\*.json')
map_paths = glob(r'D:\AIC_LITE\mapkfs\*.csv')
mets = {}
for met_path, map_path in tqdm(list(zip(met_paths, map_paths))):
    vid_code = met_path.split('\\')[-1].split('.')[0]
    # Load from metadata
    with open(met_path, 'r', encoding='utf-8') as f:
        met = json.load(f)
    yt_link = met.get('watch_url', '')

    # Load from map
    frame_map = pd.read_csv(map_path)
    for row in list(frame_map.itertuples()):
        frame_id, time_order, fps, frame_order = row.n, row.pts_time, row.fps, row.frame_idx
        code = f'{vid_code}_{frame_id:03d}'
        mets[code] = {
            'frame_id': frame_id,
            'time_order': time_order,
            'frame_order': frame_order,
            'fps': fps,
            'yt_link': f'{yt_link}&t={time_order}s'
        }

100%|██████████| 31/31 [00:00<00:00, 549.38it/s]


# Load object detection

In [5]:
obj_paths = glob(r'..\additional_data\obj_det\*.json')
objs = {}
for path, code in tqdm(list(zip(obj_paths, img_codes))):
    det = []
    with open(path, 'r') as f:
        det = json.load(f)
    objs[code] = det

100%|██████████| 8115/8115 [00:00<00:00, 12618.22it/s]


# Load transcript

In [6]:
tran_paths = glob(r'..\additional_data\transcript_img\*.txt')
trans = {}
for path, code in tqdm(list(zip(tran_paths, img_codes))):
    tran = ''
    with open(path, 'r', encoding='utf-8') as f:
        tran = f.read()
    trans[code] = tran

100%|██████████| 8115/8115 [00:32<00:00, 246.38it/s]


# Load vectors

In [17]:
siglip2_paths = glob(r'..\additional_data\feat_siglip2\*.npy')
bgem3_paths = glob(r'..\additional_data\feat_bgem3\*.npz')
vectors = {}
for code, siglip2_path, bgem3_path in tqdm(list(zip(img_codes, siglip2_paths, bgem3_paths))):
    dense_siglip2 = np.load(siglip2_path).tolist()
    all_bgem3 = np.load(bgem3_path)
    dense_bgem3 = all_bgem3['dense'].tolist()
    sparse_bgem3 = {
        'token_ids': all_bgem3['sparse_token_ids'].tolist(),
        'weights': all_bgem3['sparse_weights'].tolist()
    }
    multi = all_bgem3['colbert'].tolist()

    vectors[code] = {
        'dense_siglip2': dense_siglip2,
        'dense_bgem3': dense_bgem3,
        'sparse_bgem3': sparse_bgem3,
        'multi': multi
    }

100%|██████████| 8115/8115 [00:15<00:00, 513.18it/s]


# Update samples accordingly

In [18]:
COLLECTION_NAME = 'test'

for code in tqdm(img_codes):
    point = {
        'vector': vectors[code],
        'payload': {
            'img_path': img_path_maps[code],
            'vid_path': vid_path_maps[code],
            'frame_id': mets[code]['frame_id'],
            'frame_order': mets[code]['frame_order'],
            'time_order': mets[code]['time_order'],
            'fps': mets[code]['fps'],
            'yt_link': mets[code]['yt_link'],
            'objs': objs[code],
            'transcript': trans[code],
        }
    }
    with open(fr'..\collections\{COLLECTION_NAME}\{code}.json', 'w') as f:
        json.dump(point, f, indent=4)

100%|██████████| 8115/8115 [06:44<00:00, 20.07it/s]
